# From Cube to SLayer

**TL;DR:** SLayer can import [Cube](https://cube.dev) data models — cubes and views, in **YAML or JavaScript** — and turn them into queryable SLayer models. This notebook imports a small jaffle-shop-flavored Cube project end-to-end with `slayer import-cube`, then queries the result and checks every answer against gold SQL.

Cube import is **fully offline**: column types come from Cube's own dimension / measure declarations, so no database connection is needed to import. The DuckDB below exists only so we can *query* the imported models.

Four steps (everything generated lands in a gitignored `.cache/` next to this notebook):

1. **Build the data** — a tiny retail DuckDB (orders, customers, products) with deterministic rows.
2. **The Cube configs** — two YAML cubes, a view, and a JS cube using `FILTER_PARAMS`.
3. **Reference answers** — gold SQL, run up front.
4. **Import & query** — run `slayer import-cube`, inspect the generated models, and query them — including the `FILTER_PARAMS` pushdowns.

See also: [Importing Cube definitions](../../cube/cube_import.md) · [Variable Substitution](../14_variable_substitution/variable_substitution_nb.ipynb) — the `{var}` / `{? ?}` mechanics imported models rely on.

## Step 1 — Build the demo database

Create the retail DuckDB the cubes bind to. Cube import itself needs no database — but we query the imported models afterwards, so the data has to exist.

In [1]:
import json
import os
import sys

import pandas as pd

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_cube import (
    build_shop_duckdb,
    compute_gold,
    import_cube_cli,
    save_datasource,
    CUBE_PROJECT,
    DB_PATH,
    MODELS_DIR,
    REPORT_PATH,
    DATASOURCE_NAME,
)
from slayer.async_utils import run_sync
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.inspect.model_render import render_model_skeleton
from slayer.storage.yaml_storage import YAMLStorage

db_path = build_shop_duckdb(DB_PATH)
print(f"Built retail DuckDB '{db_path.name}' with orders, customers, products.")

Built retail DuckDB 'shop.duckdb' with orders, customers, products.


## Step 2 — The Cube configs

The project has four files. `orders` and `customers` are ordinary table-anchored cubes (with a join between them); `orders_overview` is a **view** (it owns no table); and `order_facts` is a **JavaScript** cube whose raw SQL uses `FILTER_PARAMS` — Cube's mechanism for caller-supplied filters. Watch the two `FILTER_PARAMS` lines: `category` carries `meta.required` (a *required* pushdown), `region` does not (an *optional* one).

In [2]:
for rel in [
    "model/cubes/orders.yml",
    "model/cubes/customers.yml",
    "model/views/orders_overview.yml",
    "model/cubes/order_facts.js",
]:
    print(f"# ===== {rel} =====")
    print((CUBE_PROJECT / rel).read_text().rstrip())
    print()

# ===== model/cubes/orders.yml =====
# A table-anchored Cube. Becomes a SLayer model bound to the `orders` table,
# with a join to `customers` the importer turns into a SLayer join.
cubes:
  - name: orders
    sql_table: orders
    description: One row per order line
    joins:
      - name: customers
        relationship: many_to_one
        sql: "{CUBE}.customer_id = {customers.customer_id}"
    measures:
      - name: count
        type: count
      - name: total_amount
        type: sum
        sql: "{CUBE}.amount"
        title: Total Amount
        format: currency
    dimensions:
      - name: order_id
        sql: "{CUBE}.order_id"
        type: number
        primary_key: true
      - name: status
        sql: "{CUBE}.status"
        type: string
      - name: ordered_at
        sql: "{CUBE}.ordered_at"
        type: time

# ===== model/cubes/customers.yml =====
# The join target for `orders`. Supplies the `region` dimension the view and
# the join queries group by.
cubes:
  -

## Step 3 — Reference answers (gold SQL)

Run the gold SQL **first**, up front, and stash the expected numbers. SLayer opens the DuckDB file with a read-write engine that a second raw connection can't share, so every gold query runs *before* any SLayer query touches the file.

In [3]:
GOLD = compute_gold(DB_PATH)

print("total amount (all)       :", GOLD["total"])
print("order_facts count (all)  :", GOLD["of_all"])
print("count region=North       :", GOLD["of_north"])
print("count region=South       :", GOLD["of_south"])
print("count category=Beverages :", GOLD["of_beverages"])
pd.DataFrame(GOLD["by_region"])

total amount (all)       : 1400.0
order_facts count (all)  : 6
count region=North       : 4
count region=South       : 2
count category=Beverages : 3


,region,amount
0,North,750.0
1,South,650.0


## Step 4 — Import with `slayer import-cube`

This is the one-liner an operator runs:

```bash
slayer import-cube cube_project --datasource shop_cube --storage .cache/slayer_models
```

It reads every `.yml` / `.yaml` / `.js` under the path, converts each cube and view to a SLayer model, and writes a JSON report. Below we run exactly that command (through the current interpreter) and print its summary. The two `FILTER_PARAMS` members show up as an *optional* (`region`) and a *required* (`category`) variable.

In [4]:
proc = import_cube_cli(models_dir=MODELS_DIR, report_path=REPORT_PATH)
print(proc.stdout.replace(str(REPORT_PATH.resolve()), REPORT_PATH.name).rstrip())

report = json.loads(REPORT_PATH.read_text())
errors = [i for i in report["issues"] if i["severity"] == "error"]
assert not errors, errors

Imported model: customers (3 columns, 0 measures)
Imported model: orders (4 columns, 2 measures)
Imported model: order_facts (5 columns, 2 measures)
Imported model: orders_overview (view) (3 columns, 2 measures)
  INFO [filter_params_variable/order_facts]: FILTER_PARAMS member 'region' → optional variable(s) ['region'].
  INFO [filter_params_variable/order_facts]: FILTER_PARAMS member 'category' → required variable(s) ['category'].

Done: 4 of 4 models saved (0 hidden, 1 views), 2 report issues. Report: cube_import_report.json


`import-cube` files the models under the datasource **name** but doesn't create the datasource itself. Querying needs it registered — and because the import wipes the model directory first, we register it now, *after* importing.

In [5]:
save_datasource(MODELS_DIR, DB_PATH)
print("Registered datasource:", DATASOURCE_NAME)

Registered datasource: shop_cube


## Step 5 — Inspect the generated models

`order_facts` became an **sql-mode** model whose `FILTER_PARAMS` turned into `{var}` placeholders. The model skeleton's `Variables:` line classifies them — `category (required)` vs a bare optional `region` — and `meta.cube_variables` records the per-variable contract the engine reads (`list_valued`, `required`, Cube `kind`).

In [6]:
storage = YAMLStorage(base_dir=str(MODELS_DIR))
order_facts = run_sync(storage.get_model("order_facts", data_source=DATASOURCE_NAME))

print("Model skeleton for 'order_facts':\n")
print(render_model_skeleton(model=order_facts))

print("\nmeta.cube_variables:")
for name, spec in order_facts.meta["cube_variables"].items():
    print(f"  {name}: required={spec['required']}, list_valued={spec['list_valued']}, kind={spec['kind']!r}")

Model skeleton for 'order_facts':

Columns: status, region, category, amount
Measures: count, total_amount
Aggregations: _(none)_
Joins to: _(none)_
Variables: category (required), region

meta.cube_variables:
  region: required=False, list_valued=True, kind='string'
  category: required=True, list_valued=True, kind='string'


## Query the view and the join

Point the engine at the imported models. `orders_overview` (the view) and the raw `orders` cube both reach `region` on `customers` — through the join the importer inferred from the Cube config. No SQL join is written by hand.

In [7]:
engine = SlayerQueryEngine(storage=storage)
gold_by_region = {g["region"]: g["amount"] for g in GOLD["by_region"]}

view_rows = engine.execute_sync(query={
    "source_model": "orders_overview",
    "measures": ["total_amount"],
    "dimensions": ["region"],
    "order": [{"column": "region"}],
}).data
display(pd.DataFrame(view_rows))

got = {r["orders_overview.region"]: r["orders_overview.total_amount"] for r in view_rows}
assert got == gold_by_region, f"{got} != {gold_by_region}"
print("OK — view total by region matches gold:", gold_by_region)

,orders_overview.region,orders_overview.total_amount
0,North,750.0
1,South,650.0


OK — view total by region matches gold: {'North': 750.0, 'South': 650.0}


In [8]:
join_rows = engine.execute_sync(query={
    "source_model": "orders",
    "measures": ["total_amount"],
    "dimensions": ["customers.region"],
    "order": [{"column": "customers.region"}],
}).data

got = {r["orders.customers.region"]: r["orders.total_amount"] for r in join_rows}
assert got == gold_by_region
print("OK — the raw `orders` cube reaches `customers.region` via the imported join:", got)

OK — the raw `orders` cube reaches `customers.region` via the imported join: {'North': 750.0, 'South': 650.0}


## The `FILTER_PARAMS` pushdowns

`order_facts` takes two caller-supplied filters. `category` is **required** (a bare `p.category IN ({category})`); `region` is **optional** — wrapped in a block `{? c.region IN ({region}) ?}` that collapses to `(1=1)` when omitted. Both are set-membership (`IN`) filters, so the importer marked them `list_valued`: pass a list, or a bare scalar the engine wraps into a one-element list.

In [9]:
BOTH = ["Beverages", "Bakery"]


def count(**variables):
    r = engine.execute_sync(query={
        "source_model": "order_facts",
        "measures": [{"formula": "count"}],
        "variables": variables,
    })
    return r.data[0]["order_facts.count"], r.sql


def where_lines(sql, *needles):
    return "\n".join(ln.rstrip() for ln in sql.splitlines() if any(n in ln for n in needles))

### Optional omitted → the block collapses to `(1=1)`

Supply the required `category` (both values, so nothing is excluded) and omit `region`. The optional block disappears and every order counts.

In [10]:
n, sql = count(category=BOTH)  # region omitted
print("count (region omitted):", n, " gold:", GOLD["of_all"])
assert n == GOLD["of_all"]
print("\nThe optional `region` block collapsed to (1 = 1):")
print(where_lines(sql, "1 = 1", "IN ("))

count (region omitted): 6  gold: 6

The optional `region` block collapsed to (1 = 1):
    1 = 1 AND (
      1 = 1
    ) AND p.category IN ('Beverages', 'Bakery')


### Optional supplied (a list) → a real `IN (...)`

Now pass `region=["North"]`. The block renders and filters.

In [11]:
n_north, sql = count(category=BOTH, region=["North"])
print("count region=['North']:", n_north, " gold:", GOLD["of_north"])
assert n_north == GOLD["of_north"]
print("\nRendered filter:")
print(where_lines(sql, "region", "IN ("))

count region=['North']: 4  gold: 4

Rendered filter:
    c.region,
      c.region IN ('North')
    ) AND p.category IN ('Beverages', 'Bakery')


### A scalar for a list-valued variable → coerced to a one-element list

The importer wrote the `IN (...)` parentheses, so a caller has nowhere to put per-element quotes. SLayer therefore wraps a bare scalar into a one-element list — `region="North"` means exactly `region=["North"]`.

In [12]:
n_scalar, _ = count(category=BOTH, region="North")   # bare string, not a list
print("region='North' (scalar):", n_scalar, "  region=['North'] (list):", n_north)
assert n_scalar == n_north

n_bev, _ = count(category="Beverages")   # scalar for the required var; region omitted
print("category='Beverages' (scalar):", n_bev, " gold:", GOLD["of_beverages"])
assert n_bev == GOLD["of_beverages"]

region='North' (scalar): 4   region=['North'] (list): 4
category='Beverages' (scalar): 3  gold: 3


### The required pushdown is not optional

Omit `category` (supplying only `region`) and the query raises, naming the missing variable — a required pushdown fails loudly rather than matching nothing.

In [13]:
try:
    count(region=["North"])   # category (required) omitted
except ValueError as e:
    print("Omitting the required `category` raises:")
    print(" ", e)

Omitting the required `category` raises:
  Undefined variable 'category' in filter: '\n      SELECT o.order_id, o.amount, o.status, c.region, p.category\n      FROM orders o\n      LEFT JOIN customers c ON o.customer_id = c.customer_id\n      LEFT JOIN products  p ON o.product_id  = p.product_id\n      WHERE 1 = 1\n          AND {? c.region IN ({region}) ?}\n          AND p.category IN ({category})\n    '. Available variables: ['region']


## Recap

Starting from a Cube project we did not write, `slayer import-cube`:

- turned two YAML cubes and a view into queryable SLayer models, with a join inferred from the Cube config,
- converted a JS cube's `FILTER_PARAMS` into SLayer `{var}` substitution — a **required** pushdown (`category`) and an **optional** one (`region`) that collapses to `(1=1)` when omitted,
- marked both set-membership pushdowns `list_valued`, so a bare scalar is coerced to a one-element list.

Every answer matched gold SQL.

### Further reading

- [Importing Cube definitions](../../cube/cube_import.md) — the full conversion reference (what maps, what fails cleanly, the report).
- [Variable Substitution](../14_variable_substitution/variable_substitution_nb.ipynb) — the `{var}` / `{? ?}` mechanics, across all of SLayer's raw-SQL surfaces.